In [1]:
#预处理
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [2]:
#定义训练参数
from transformers import TrainingArguments
#必传参数是目录,push_to_hub=True 自动将模型上传到 Hub
training_args = TrainingArguments("outputs\\day05-savetrain",push_to_hub=True)

In [3]:
#定义模型
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
#定义一个 Trainer
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

C:\Users\27729\AppData\Local\Temp\ipykernel_13228\1996419550.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
#训练模型
trainer.train()

In [5]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

(408, 2) (408,)


In [6]:
import numpy as np
preds = np.argmax(predictions.predictions, axis=-1)

In [7]:
import evaluate
metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.6813725490196079, 'f1': 0.8104956268221575}

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=AutoModelForSequenceClassification.from_pretrained("outputs/day05-savetrain/checkpoint-1377"),
    args=TrainingArguments("outputs/day05-savetrain"),
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

import evaluate
import numpy as np

# 加载 MRPC 评估器
metric = evaluate.load("glue", "mrpc")

# 获取模型预测
predictions = trainer.predict(tokenized_datasets["validation"])

# logits → 标签（取分数高的那一类）
preds = np.argmax(predictions.predictions, axis=-1)

# 计算指标
result = metric.compute(predictions=preds, references=predictions.label_ids)
print(result)
# → {'accuracy': 0.XXX, 'f1': 0.XXX}


{'accuracy': 0.8333333333333334, 'f1': 0.8847457627118644}


In [13]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments("notebooks\\outputs\\day05-savetrain2", eval_strategy="epoch")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\27729\AppData\Local\Temp\ipykernel_13228\762194472.py:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.421247,0.840686,0.888508
2,0.554600,0.406002,0.848039,0.891986
3,0.351600,0.585907,0.860294,0.904202


TrainOutput(global_step=1377, training_loss=0.38776313488086933, metrics={'train_runtime': 236.0162, 'train_samples_per_second': 46.624, 'train_steps_per_second': 5.834, 'total_flos': 405114969714960.0, 'train_loss': 0.38776313488086933, 'epoch': 3.0})